In [ ]:
# Instalar e importar
!pip install requests beautifulsoup4 lxml pandas tqdm --quiet

import os
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Parámetros ───────────────────────────────────────────────────────────────
AÑO_INICIO = 2026
AÑO_FIN    = 2026

AÑOS    = list(range(AÑO_INICIO, AÑO_FIN + 1))
WORKERS = 8
REINTENTOS = 4          # reintentos por petición (backoff exponencial)

# Carpeta de Google Drive donde se guardará el CSV (ID de la URL de la carpeta)
DRIVE_FOLDER_ID = "1LOErZkBcpWPnT4b_w3LJ7yNQG5ruTSFJ"

# ── Tablas de referencia (SOLO LORETO, SAN MARTÍN Y UCAYALI) ─────────────────
dept = pd.DataFrame({
    'codigo': ['16', '22', '25'],
    'nombre': ['16: LORETO', '22: SAN MARTIN', '25: UCAYALI']
})

nivel = pd.DataFrame({
    'codigo': ['E', 'M', 'R'],
    'nombre': ['GOBIERNO NACIONAL', 'GOBIERNOS LOCALES', 'GOBIERNOS REGIONALES']
})

dept_map  = dict(zip(dept['codigo'],  dept['nombre']))
nivel_map = dict(zip(nivel['codigo'], nivel['nombre']))

combinaciones = list(product(AÑOS, dept['codigo'], nivel['codigo']))
print(f"Total de combinaciones: {len(combinaciones)}")

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})

# ── Funciones de extracción ──────────────────────────────────────────────────
def construir_url(año, d, n):
    return (
        "https://apps5.mineco.gob.pe/transparencia/mensual/"
        f"Navegar_6.aspx?&0=&24=6-2-6&21={d}&1={n}&8=&y={año}"
    )

def descargar_tabla(año, d, n):
    """Descarga la tabla 'Data'. Devuelve None si la página no tiene tabla
    (combinación sin datos). Si hay error de red/HTTP reintenta con backoff
    y, si agota los intentos, relanza la excepción (no se oculta)."""
    url = construir_url(año, d, n)
    for i in range(REINTENTOS):
        try:
            resp = SESSION.get(url, timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.content, 'lxml')
            tabla_html = soup.find('table', class_='Data')
            if tabla_html is None:
                return None
            return pd.read_html(StringIO(str(tabla_html)))[0]
        except Exception:
            if i == REINTENTOS - 1:
                raise
            time.sleep((2 ** i) + random.random())

def procesar_tabla(raw):
    """Separa la fila 'Total' que muestra Consulta Amigable de las filas por
    función. Devuelve (df_funciones[FUNCION, PIM], total_reportado)."""
    raw = raw.copy()
    if raw.shape[1] < 4:
        return None, None

    cols = list(raw.columns)
    cols[1] = 'FUNCION'
    cols[3] = 'PIM'
    raw.columns = cols

    # Detectar fila de total (la etiqueta puede estar en la col. 0 o en FUNCION)
    etiqueta = raw.iloc[:, 0].astype(str) + ' ' + raw['FUNCION'].astype(str)
    es_total = etiqueta.str.upper().str.contains(r'\bTOTAL\b', regex=True)

    total = None
    if es_total.any():
        t = pd.to_numeric(
            raw.loc[es_total, 'PIM'].astype(str).str.replace(',', ''),
            errors='coerce').dropna()
        if not t.empty:
            total = float(t.iloc[0])

    df = raw.loc[~es_total, ['FUNCION', 'PIM']].dropna().copy()
    df['PIM'] = pd.to_numeric(df['PIM'].astype(str).str.replace(',', ''),
                              errors='coerce')
    df = df.dropna()
    return (df if not df.empty else None), total

def extraer_datos(año, d, n):
    """Devuelve dict con las funciones, el total reportado y el posible error."""
    try:
        raw = descargar_tabla(año, d, n)
        if raw is None:
            return {'clave': (año, d, n), 'df': None, 'total': None, 'error': None}
        df, total = procesar_tabla(raw)
        if df is not None:
            df['AÑO']               = año
            df['DEPARTAMENTO']      = dept_map[d]
            df['NIVEL DE GOBIERNO'] = nivel_map[n]
            df = df[['AÑO', 'DEPARTAMENTO', 'NIVEL DE GOBIERNO', 'FUNCION', 'PIM']]
        return {'clave': (año, d, n), 'df': df, 'total': total, 'error': None}
    except Exception as e:
        return {'clave': (año, d, n), 'df': None, 'total': None, 'error': str(e)}

# ── Ejecución en paralelo ────────────────────────────────────────────────────
resultados, totales_reportados, fallos = [], {}, []

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futuros = [executor.submit(extraer_datos, a, d, n) for a, d, n in combinaciones]
    for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Extrayendo datos"):
        r = futuro.result()
        if r['error']:
            fallos.append((r['clave'], r['error']))
        if r['df'] is not None and not r['df'].empty:
            resultados.append(r['df'])
        if r['total'] is not None:
            totales_reportados[r['clave']] = r['total']

if fallos:
    print(f"⚠️ {len(fallos)} combinaciones fallaron tras {REINTENTOS} intentos:")
    for (a, d, n), err in fallos:
        print(f"   - {a} | {dept_map[d]} | {nivel_map[n]} -> {err[:100]}")

if resultados:
    resultado_final = pd.concat(resultados, ignore_index=True)
    print(f"\n✅ Filas obtenidas: {len(resultado_final):,}")
    display(resultado_final.head())
else:
    print("⚠️ No se obtuvieron datos.")

# ── Subida automática a Google Drive ─────────────────────────────────────────
def subir_a_drive(ruta_local, folder_id):
    """Sube el archivo a la carpeta de Drive indicada (por ID). Si ya existe un
    archivo con el mismo nombre en esa carpeta, lo sobrescribe en lugar de
    duplicarlo. Devuelve el enlace al archivo."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    auth.authenticate_user()   # usa la cuenta con acceso a la carpeta
    drive = build('drive', 'v3', cache_discovery=False)

    nombre = os.path.basename(ruta_local)
    consulta = (f"name = '{nombre}' and '{folder_id}' in parents "
                f"and trashed = false")
    existentes = drive.files().list(
        q=consulta, fields='files(id, name)',
        supportsAllDrives=True, includeItemsFromAllDrives=True
    ).execute().get('files', [])

    media = MediaFileUpload(ruta_local, mimetype='text/csv', resumable=True)
    if existentes:
        archivo = drive.files().update(
            fileId=existentes[0]['id'], media_body=media,
            fields='id, webViewLink', supportsAllDrives=True
        ).execute()
        accion = 'actualizado (se sobrescribió el existente)'
    else:
        archivo = drive.files().create(
            body={'name': nombre, 'parents': [folder_id]},
            media_body=media, fields='id, webViewLink', supportsAllDrives=True
        ).execute()
        accion = 'creado'
    print(f"☁️ Archivo {accion} en Drive: {archivo.get('webViewLink')}")
    return archivo.get('webViewLink')

# ── Guardar y descargar CSV ──────────────────────────────────────────────────
if resultados:
    from google.colab import files

    sufijo = "LOR - SMN - UCA"
    if AÑO_INICIO == AÑO_FIN:
        nombre_archivo = f"PIM Funciones {AÑO_INICIO} - {sufijo}.csv"
    else:
        nombre_archivo = f"PIM Funciones {AÑO_INICIO}-{AÑO_FIN} - {sufijo}.csv"

    resultado_final.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f"✅ Archivo guardado localmente: {nombre_archivo}")

    try:
        subir_a_drive(nombre_archivo, DRIVE_FOLDER_ID)
    except Exception as e:
        print(f"❌ No se pudo subir a Drive: {e}")
        print("   Se descarga el archivo a tu computadora como respaldo.")
        files.download(nombre_archivo)
else:
    print("⚠️ No hay datos para guardar.")

Total de combinaciones: 9


Extrayendo datos:   0%|          | 0/9 [00:00<?, ?it/s]


✅ Filas obtenidas: 160


,AÑO,DEPARTAMENTO,NIVEL DE GOBIERNO,FUNCION,PIM
0,2026,25: UCAYALI,GOBIERNOS LOCALES,"03: PLANEAMIENTO, GESTION Y RESERVA DE CONTING...",25593639
1,2026,25: UCAYALI,GOBIERNOS LOCALES,05: ORDEN PUBLICO Y SEGURIDAD,30598106
2,2026,25: UCAYALI,GOBIERNOS LOCALES,07: TRABAJO,136541
3,2026,25: UCAYALI,GOBIERNOS LOCALES,08: COMERCIO,448071
4,2026,25: UCAYALI,GOBIERNOS LOCALES,09: TURISMO,902879


✅ Archivo guardado localmente: PIM Funciones 2026 - LOR - SMN - UCA.csv


☁️ Archivo creado en Drive: https://drive.google.com/file/d/1K4gJ8me82u2kE4n9KHCozeGU4A8rL11l/view?usp=drivesdk
